# Experiment: BTC Transformer Confirmatory Study

Этот ноутбук предназначен для финального подтверждающего этапа перед суммаризацией результатов в курсовой.

Здесь больше нет широкого поиска. Мы проверяем только те конфигурации, которые уже показали лучший сигнал во втором цикле:

Регрессия:
- `15m`, горизонт `12`: `zero_return`, `numeric_relative`, `symbolic_no_volume`
- `1h`, горизонт `12`: `zero_return`, `numeric_relative`, `symbolic_fine_body`
- `1h`, горизонт `1`: `zero_return`, `numeric_relative`, `symbolic_no_volume`

Классификация направления:
- `15m`, горизонт `4`: `majority_sign`, `last_sign`, `numeric_relative`, `symbolic_fine_body`
- `1h`, горизонт `1`: `majority_sign`, `last_sign`, `numeric_relative`, `symbolic_no_volume`

Цель:
- получить mean/std по фолдам
- подтвердить, что найденные конфигурации действительно устойчивы
- подготовить компактные таблицы для текста курсовой

## Что будет на выходе

Ноутбук сохранит в `OUT_DIR`:
- `regression_fold_metrics.csv`
- `direction_fold_metrics.csv`
- `regression_summary.csv`
- `direction_summary.csv`
- `paper_ready_table.csv`
- `regression_predictions.csv`
- `direction_predictions.csv`

Основные столбцы для интерпретации:
- регрессия: `price_mae_mean`, `price_mae_std`, `price_mae_vs_baseline_mean`
- направление: `balanced_accuracy_mean`, `balanced_accuracy_std`, `balanced_accuracy_vs_baseline_mean`

In [ ]:
import os
import math
import time
import copy
import random
from dataclasses import dataclass, asdict
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from IPython.display import display


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
RUN_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = os.path.join(RUN_ROOT, "data")
OUT_DIR = os.path.join(RUN_ROOT, "btc_transformer_confirmatory_runs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("RUN_ROOT:", RUN_ROOT)
print("OUT_DIR:", OUT_DIR)

In [ ]:
@dataclass
class CFG:
    days_1m: int = 120
    providers: tuple = ("coinbase", "bitstamp", "kraken")
    max_folds: int = 3

    batch_size: int = 128
    regression_epochs: int = 5
    direction_epochs: int = 5
    patience: int = 2
    lr: float = 3e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    amp: bool = True
    num_workers: int = 0
    pin_memory: bool = False

    d_model: int = 80
    n_heads: int = 4
    n_layers: int = 2
    dropout: float = 0.1

    relative_feature_cols: tuple = (
        "logret_1",
        "logret_4",
        "logret_season",
        "open_gap",
        "close_open",
        "high_open",
        "low_open",
        "hl_range",
        "body_ratio",
        "upper_ratio",
        "lower_ratio",
        "vol_dev",
        "rolling_vol",
    )


cfg = CFG()

FREQ_PRESETS = {
    "15m": {
        "resample_rule": "15min",
        "lookback": 128,
        "min_train": 3600,
        "val_size": 672,
        "test_size": 672,
        "step": 672,
        "seasonal_lag": 96,
    },
    "1h": {
        "resample_rule": "1h",
        "lookback": 96,
        "min_train": 1200,
        "val_size": 336,
        "test_size": 336,
        "step": 336,
        "seasonal_lag": 24,
    },
}

TOKENIZER_SPECS = {
    "no_volume": {
        "body_bins": (0.05, 0.20, 0.50),
        "wick_bins": (0.35,),
        "vol_bins": (),
        "include_volume": False,
    },
    "fine_body": {
        "body_bins": (0.03, 0.12, 0.30, 0.55),
        "wick_bins": (0.18, 0.45),
        "vol_bins": (0.70, 1.00, 1.35),
        "include_volume": True,
    },
}

REGRESSION_RUNS = [
    {"freq": "15m", "horizon": 12, "learned_models": ["numeric_relative", "symbolic_no_volume"], "baseline": "zero_return"},
    {"freq": "1h", "horizon": 12, "learned_models": ["numeric_relative", "symbolic_fine_body"], "baseline": "zero_return"},
    {"freq": "1h", "horizon": 1, "learned_models": ["numeric_relative", "symbolic_no_volume"], "baseline": "zero_return"},
]

DIRECTION_RUNS = [
    {"freq": "15m", "horizon": 4, "learned_models": ["numeric_relative", "symbolic_fine_body"], "baselines": ["majority_sign", "last_sign"], "baseline": "majority_sign"},
    {"freq": "1h", "horizon": 1, "learned_models": ["numeric_relative", "symbolic_no_volume"], "baselines": ["majority_sign", "last_sign"], "baseline": "majority_sign"},
]

display(pd.Series(asdict(cfg), name="value"))
display(pd.DataFrame(REGRESSION_RUNS))
display(pd.DataFrame(DIRECTION_RUNS))

In [ ]:
def _read_cache(cache_path: str) -> pd.DataFrame:
    df = pd.read_csv(cache_path)
    df["Datetime"] = pd.to_datetime(df["Datetime"], utc=True)
    return df.sort_values("Datetime").drop_duplicates(subset=["Datetime"]).reset_index(drop=True)


def _dt_utc_now_floor_min():
    return datetime.now(timezone.utc).replace(second=0, microsecond=0)


def _iso(dt: datetime) -> str:
    return dt.astimezone(timezone.utc).isoformat()


def download_coinbase_1m(days: int, cache_path: str, max_retries: int = 6, sleep_base: float = 1.0):
    if os.path.exists(cache_path):
        return _read_cache(cache_path)

    end_dt = _dt_utc_now_floor_min()
    start_dt = end_dt - timedelta(days=days)
    print(f"Downloading Coinbase BTC-USD 1m: {start_dt.isoformat()} -> {end_dt.isoformat()}")

    url = "https://api.exchange.coinbase.com/products/BTC-USD/candles"
    headers = {"Accept": "application/json"}
    chunk = timedelta(minutes=300)
    rows = []
    cur = start_dt

    while cur < end_dt:
        chunk_end = min(end_dt, cur + chunk)
        params = {"granularity": 60, "start": _iso(cur), "end": _iso(chunk_end)}
        attempt = 0
        while True:
            try:
                r = requests.get(url, headers=headers, params=params, timeout=30)
                if r.status_code != 200:
                    raise RuntimeError(f"Coinbase HTTP {r.status_code}: {r.text[:200]}")
                data = r.json()
                break
            except Exception:
                attempt += 1
                if attempt > max_retries:
                    raise
                time.sleep(sleep_base * (2 ** (attempt - 1)))

        for candle in data:
            ts = int(candle[0])
            rows.append([ts, float(candle[3]), float(candle[2]), float(candle[1]), float(candle[4]), float(candle[5])])

        cur = chunk_end
        time.sleep(0.12)

    df = pd.DataFrame(rows, columns=["EpochSec", "Open", "High", "Low", "Close", "Volume"])
    df["Datetime"] = pd.to_datetime(df["EpochSec"], unit="s", utc=True)
    df = df.drop(columns=["EpochSec"]).sort_values("Datetime").drop_duplicates(subset=["Datetime"]).reset_index(drop=True)
    df.to_csv(cache_path, index=False)
    return df


def download_bitstamp_1m(days: int, cache_path: str, max_retries: int = 6, sleep_base: float = 1.0):
    if os.path.exists(cache_path):
        return _read_cache(cache_path)

    end_dt = _dt_utc_now_floor_min()
    start_dt = end_dt - timedelta(days=days)
    print(f"Downloading Bitstamp btcusd 1m: {start_dt.isoformat()} -> {end_dt.isoformat()}")

    url = "https://www.bitstamp.net/api/v2/ohlc/btcusd/"
    cur = int(start_dt.timestamp())
    end_sec = int(end_dt.timestamp())
    step = 60
    limit = 1000
    rows = []

    while cur < end_sec:
        attempt = 0
        while True:
            try:
                params = {"step": step, "limit": limit, "start": cur}
                r = requests.get(url, params=params, timeout=30)
                if r.status_code != 200:
                    raise RuntimeError(f"Bitstamp HTTP {r.status_code}: {r.text[:200]}")
                data = r.json().get("data", {}).get("ohlc", [])
                break
            except Exception:
                attempt += 1
                if attempt > max_retries:
                    raise
                time.sleep(sleep_base * (2 ** (attempt - 1)))

        if not data:
            cur += step * limit
            continue

        for candle in data:
            ts = int(candle["timestamp"])
            rows.append([ts, float(candle["open"]), float(candle["high"]), float(candle["low"]), float(candle["close"]), float(candle["volume"])])

        cur = int(data[-1]["timestamp"]) + step
        time.sleep(0.12)

    df = pd.DataFrame(rows, columns=["EpochSec", "Open", "High", "Low", "Close", "Volume"])
    df["Datetime"] = pd.to_datetime(df["EpochSec"], unit="s", utc=True)
    df = df.drop(columns=["EpochSec"]).sort_values("Datetime").drop_duplicates(subset=["Datetime"]).reset_index(drop=True)
    df.to_csv(cache_path, index=False)
    return df


def download_kraken_1m(days: int, cache_path: str, max_retries: int = 6, sleep_base: float = 1.0):
    if os.path.exists(cache_path):
        return _read_cache(cache_path)

    end_dt = _dt_utc_now_floor_min()
    start_dt = end_dt - timedelta(days=days)
    print(f"Downloading Kraken XBTUSD 1m: {start_dt.isoformat()} -> {end_dt.isoformat()}")

    url = "https://api.kraken.com/0/public/OHLC"
    pair = "XBTUSD"
    since = int(start_dt.timestamp())
    end_sec = int(end_dt.timestamp())
    rows = []

    while since < end_sec:
        attempt = 0
        while True:
            try:
                params = {"pair": pair, "interval": 1, "since": since}
                r = requests.get(url, params=params, timeout=30)
                if r.status_code != 200:
                    raise RuntimeError(f"Kraken HTTP {r.status_code}: {r.text[:200]}")
                js = r.json()
                if js.get("error"):
                    raise RuntimeError(f"Kraken error: {js['error']}")
                data = js["result"][pair]
                new_since = int(js["result"]["last"])
                break
            except Exception:
                attempt += 1
                if attempt > max_retries:
                    raise
                time.sleep(sleep_base * (2 ** (attempt - 1)))

        for candle in data:
            ts = int(candle[0])
            if int(start_dt.timestamp()) <= ts <= end_sec:
                rows.append([ts, float(candle[1]), float(candle[2]), float(candle[3]), float(candle[4]), float(candle[6])])

        since = new_since if new_since > since else since + 60 * 720
        time.sleep(0.12)

    df = pd.DataFrame(rows, columns=["EpochSec", "Open", "High", "Low", "Close", "Volume"])
    df["Datetime"] = pd.to_datetime(df["EpochSec"], unit="s", utc=True)
    df = df.drop(columns=["EpochSec"]).sort_values("Datetime").drop_duplicates(subset=["Datetime"]).reset_index(drop=True)
    df.to_csv(cache_path, index=False)
    return df


def load_1m_with_fallback(cfg: CFG) -> pd.DataFrame:
    last_err = None
    for provider in cfg.providers:
        cache_path = os.path.join(DATA_DIR, f"{provider.upper()}_BTC_1m_last{cfg.days_1m}d.csv")
        try:
            if provider == "coinbase":
                return download_coinbase_1m(cfg.days_1m, cache_path)
            if provider == "bitstamp":
                return download_bitstamp_1m(cfg.days_1m, cache_path)
            if provider == "kraken":
                return download_kraken_1m(cfg.days_1m, cache_path)
            raise ValueError(f"Unknown provider: {provider}")
        except Exception as e:
            print(f"[{provider}] failed: {repr(e)[:200]}")
            last_err = e
    raise RuntimeError(f"All providers failed. Last error: {repr(last_err)}")

In [ ]:
def resample_ohlcv(df: pd.DataFrame, rule: str) -> pd.DataFrame:
    out = df.copy()
    out["Datetime"] = pd.to_datetime(out["Datetime"], utc=True)
    out = out.sort_values("Datetime").set_index("Datetime")
    resampled = pd.concat(
        [
            out["Open"].resample(rule).first(),
            out["High"].resample(rule).max(),
            out["Low"].resample(rule).min(),
            out["Close"].resample(rule).last(),
            out["Volume"].resample(rule).sum(),
        ],
        axis=1,
    )
    resampled.columns = ["Open", "High", "Low", "Close", "Volume"]
    return resampled.dropna()


def add_relative_features(df: pd.DataFrame, seasonal_lag: int) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-9
    prev_close = out["Close"].shift(1).clip(lower=eps)
    out["range"] = (out["High"] - out["Low"]).clip(lower=eps)
    out["body"] = out["Close"] - out["Open"]
    out["body_abs"] = out["body"].abs()
    out["upper_wick"] = (out["High"] - out[["Open", "Close"]].max(axis=1)).clip(lower=0.0)
    out["lower_wick"] = (out[["Open", "Close"]].min(axis=1) - out["Low"]).clip(lower=0.0)
    out["body_ratio"] = out["body_abs"] / out["range"]
    out["upper_ratio"] = out["upper_wick"] / out["range"]
    out["lower_ratio"] = out["lower_wick"] / out["range"]

    out["logret_1"] = np.log(out["Close"].clip(lower=eps) / prev_close).fillna(0.0)
    out["logret_4"] = np.log(out["Close"].clip(lower=eps) / out["Close"].shift(4).clip(lower=eps))
    out["logret_season"] = np.log(out["Close"].clip(lower=eps) / out["Close"].shift(seasonal_lag).clip(lower=eps))
    out["open_gap"] = np.log(out["Open"].clip(lower=eps) / prev_close)
    out["close_open"] = np.log(out["Close"].clip(lower=eps) / out["Open"].clip(lower=eps))
    out["high_open"] = np.log(out["High"].clip(lower=eps) / out["Open"].clip(lower=eps))
    out["low_open"] = np.log(out["Low"].clip(lower=eps) / out["Open"].clip(lower=eps))
    out["hl_range"] = np.log(out["High"].clip(lower=eps) / out["Low"].clip(lower=eps))

    log_volume = np.log1p(out["Volume"].clip(lower=0.0))
    out["vol_dev"] = log_volume - log_volume.rolling(seasonal_lag, min_periods=max(8, seasonal_lag // 4)).median()
    out["vol_ratio"] = out["Volume"] / out["Volume"].rolling(seasonal_lag, min_periods=max(8, seasonal_lag // 4)).median().clip(lower=eps)
    out["rolling_vol"] = out["logret_1"].rolling(seasonal_lag, min_periods=max(8, seasonal_lag // 4)).std()

    out = out.replace([np.inf, -np.inf], np.nan).dropna()
    return out


def _bucket(value: float, thresholds, prefix: str) -> str:
    for idx, thr in enumerate(thresholds):
        if value < thr:
            return f"{prefix}{idx}"
    return f"{prefix}{len(thresholds)}"


def candle_direction(body: float, body_ratio: float) -> str:
    if body_ratio < 0.04:
        return "doji"
    return "up" if body > 0 else "down"


def candle_token_from_row(row: pd.Series, spec: dict) -> str:
    parts = [
        candle_direction(float(row["body"]), float(row["body_ratio"])),
        _bucket(float(row["body_ratio"]), spec["body_bins"], "b"),
        _bucket(float(row["upper_ratio"]), spec["wick_bins"], "u"),
        _bucket(float(row["lower_ratio"]), spec["wick_bins"], "l"),
    ]
    if spec["include_volume"]:
        parts.append(_bucket(float(row["vol_ratio"]), spec["vol_bins"], "v"))
    return "|".join(parts)


def build_frame(raw_1m: pd.DataFrame, freq: str) -> pd.DataFrame:
    preset = FREQ_PRESETS[freq]
    frame = resample_ohlcv(raw_1m, preset["resample_rule"])
    frame = add_relative_features(frame, preset["seasonal_lag"])
    for variant, spec in TOKENIZER_SPECS.items():
        frame[f"token_{variant}"] = [candle_token_from_row(row, spec) for _, row in frame.iterrows()]
    return frame


def make_folds(n_points: int, freq: str, cfg: CFG):
    preset = FREQ_PRESETS[freq]
    max_horizon = max([r["horizon"] for r in REGRESSION_RUNS + DIRECTION_RUNS if r["freq"] == freq])
    train_end = max(preset["min_train"], preset["lookback"] + max_horizon + 2)
    folds = []
    while train_end + preset["val_size"] + preset["test_size"] <= n_points:
        folds.append(
            {
                "train_end": train_end,
                "val_end": train_end + preset["val_size"],
                "test_end": train_end + preset["val_size"] + preset["test_size"],
            }
        )
        train_end += preset["step"]
    return folds[: cfg.max_folds]


def make_anchors(start: int, end: int, lookback: int, horizon: int):
    first = max(start, lookback)
    last_exclusive = end - horizon + 1
    if last_exclusive <= first:
        return np.array([], dtype=np.int64)
    return np.arange(first, last_exclusive, dtype=np.int64)


def fit_feature_scaler(train_df: pd.DataFrame, feature_cols):
    scaler = {}
    for col in feature_cols:
        values = train_df[col].values.astype(np.float32)
        scaler[col] = (float(np.nanmean(values)), float(np.nanstd(values) + 1e-6))
    return scaler


def fit_return_scaler(close_values: np.ndarray, anchors: np.ndarray, horizon: int):
    last_close = np.clip(close_values[anchors - 1], 1e-8, None)
    future_close = np.clip(close_values[anchors + horizon - 1], 1e-8, None)
    targets = np.log(future_close) - np.log(last_close)
    return float(np.mean(targets)), float(np.std(targets) + 1e-6)


def inverse_scale(values: np.ndarray, scaler):
    mean, std = scaler
    return values * std + mean


def fit_token_vocab(train_tokens):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token in pd.Series(train_tokens).astype(str).unique():
        if token not in vocab:
            vocab[token] = len(vocab)
    return vocab

In [ ]:
class NumericDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, feature_cols, anchors, lookback: int, horizon: int, task: str, x_scaler=None, y_scaler=None):
        self.features = frame[list(feature_cols)].values.astype(np.float32)
        self.close = frame["Close"].values.astype(np.float32)
        self.anchors = np.asarray(anchors, dtype=np.int64)
        self.lookback = lookback
        self.horizon = horizon
        self.task = task
        self.feature_cols = list(feature_cols)
        self.x_scaler = x_scaler
        self.y_scaler = y_scaler

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        anchor = int(self.anchors[idx])
        x = self.features[anchor - self.lookback : anchor].copy()
        if self.x_scaler is not None:
            for j, col in enumerate(self.feature_cols):
                mean, std = self.x_scaler[col]
                x[:, j] = (x[:, j] - mean) / std

        last_close = float(self.close[anchor - 1])
        future_close = float(self.close[anchor + self.horizon - 1])
        future_return = math.log(max(future_close, 1e-8)) - math.log(max(last_close, 1e-8))

        if self.task == "regression":
            mean, std = self.y_scaler
            target = (future_return - mean) / std
        else:
            target = 1.0 if future_return > 0 else 0.0

        return (
            torch.from_numpy(x),
            torch.tensor(target, dtype=torch.float32),
            torch.tensor(anchor, dtype=torch.long),
            torch.tensor(last_close, dtype=torch.float32),
            torch.tensor(future_close, dtype=torch.float32),
            torch.tensor(future_return, dtype=torch.float32),
        )


class TokenDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, token_col: str, token_vocab: dict, anchors, lookback: int, horizon: int, task: str, y_scaler=None):
        self.tokens = frame[token_col].astype(str).tolist()
        self.vocab = token_vocab
        self.close = frame["Close"].values.astype(np.float32)
        self.anchors = np.asarray(anchors, dtype=np.int64)
        self.lookback = lookback
        self.horizon = horizon
        self.task = task
        self.y_scaler = y_scaler

    def __len__(self):
        return len(self.anchors)

    def __getitem__(self, idx):
        anchor = int(self.anchors[idx])
        token_ids = np.array([self.vocab.get(token, 1) for token in self.tokens[anchor - self.lookback : anchor]], dtype=np.int64)
        last_close = float(self.close[anchor - 1])
        future_close = float(self.close[anchor + self.horizon - 1])
        future_return = math.log(max(future_close, 1e-8)) - math.log(max(last_close, 1e-8))

        if self.task == "regression":
            mean, std = self.y_scaler
            target = (future_return - mean) / std
        else:
            target = 1.0 if future_return > 0 else 0.0

        return (
            torch.from_numpy(token_ids),
            torch.tensor(target, dtype=torch.float32),
            torch.tensor(anchor, dtype=torch.long),
            torch.tensor(last_close, dtype=torch.float32),
            torch.tensor(future_close, dtype=torch.float32),
            torch.tensor(future_return, dtype=torch.float32),
        )


def make_loader(dataset: Dataset, shuffle: bool, cfg: CFG):
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 10000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model, dtype=torch.float32)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, : x.size(1)])


class NumericTransformer(nn.Module):
    def __init__(self, n_features: int, cfg: CFG, max_len: int):
        super().__init__()
        self.input_proj = nn.Linear(n_features, cfg.d_model)
        self.pos = PositionalEncoding(cfg.d_model, dropout=cfg.dropout, max_len=max_len + 8)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, 2 * cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(2 * cfg.d_model, 1),
        )

    def forward(self, x):
        z = self.input_proj(x)
        z = self.pos(z)
        z = self.encoder(z)
        return self.head(z[:, -1, :]).squeeze(-1)


class TokenTransformer(nn.Module):
    def __init__(self, vocab_size: int, cfg: CFG, max_len: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, cfg.d_model, padding_idx=0)
        self.pos = PositionalEncoding(cfg.d_model, dropout=cfg.dropout, max_len=max_len + 8)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=cfg.d_model,
            nhead=cfg.n_heads,
            dim_feedforward=4 * cfg.d_model,
            dropout=cfg.dropout,
            activation="gelu",
            batch_first=True,
            norm_first=False,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=cfg.n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(cfg.d_model),
            nn.Linear(cfg.d_model, 2 * cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(2 * cfg.d_model, 1),
        )

    def forward(self, x):
        z = self.embedding(x)
        z = self.pos(z)
        z = self.encoder(z)
        return self.head(z[:, -1, :]).squeeze(-1)


def build_model(model_name: str, cfg: CFG, max_len: int, n_features=None, vocab_size=None):
    if model_name == "numeric_relative":
        return NumericTransformer(n_features, cfg, max_len).to(device)
    if model_name.startswith("symbolic_"):
        return TokenTransformer(vocab_size, cfg, max_len).to(device)
    raise ValueError(f"Unknown model_name: {model_name}")

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn, scaler_amp, cfg: CFG):
    model.train()
    losses = []
    for xb, yb, *_ in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast("cuda", enabled=(cfg.amp and device == "cuda")):
            pred = model(xb)
            loss = loss_fn(pred, yb)
        scaler_amp.scale(loss).backward()
        if cfg.grad_clip:
            scaler_amp.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler_amp.step(optimizer)
        scaler_amp.update()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def eval_epoch(model, loader, loss_fn, cfg: CFG):
    model.eval()
    losses = []
    with torch.no_grad():
        for xb, yb, *_ in loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            with autocast("cuda", enabled=(cfg.amp and device == "cuda")):
                pred = model(xb)
                loss = loss_fn(pred, yb)
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


def fit_model(model_name: str, task: str, train_ds: Dataset, val_ds: Dataset, cfg: CFG, max_len: int, n_features=None, vocab_size=None):
    model = build_model(model_name, cfg, max_len=max_len, n_features=n_features, vocab_size=vocab_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    loss_fn = nn.HuberLoss(delta=1.0) if task == "regression" else nn.BCEWithLogitsLoss()
    scaler_amp = GradScaler("cuda", enabled=(cfg.amp and device == "cuda"))
    train_loader = make_loader(train_ds, shuffle=True, cfg=cfg)
    val_loader = make_loader(val_ds, shuffle=False, cfg=cfg)

    max_epochs = cfg.regression_epochs if task == "regression" else cfg.direction_epochs
    best_val = float("inf")
    best_state = None
    wait = 0
    history = []

    for epoch in range(1, max_epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, loss_fn, scaler_amp, cfg)
        val_loss = eval_epoch(model, val_loader, loss_fn, cfg)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        print(f"{task:>10s} | {model_name:>18s} | epoch {epoch:02d} | train={train_loss:.6f} | val={val_loss:.6f}")

        if val_loss < best_val - 1e-6:
            best_val = val_loss
            best_state = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})
            wait = 0
        else:
            wait += 1
            if wait >= cfg.patience:
                break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


def rmse(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.sqrt(np.mean((a - b) ** 2)))


def mae(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    return float(np.mean(np.abs(a - b)))


def accuracy_np(y_true, y_pred):
    return float(np.mean(np.asarray(y_true, dtype=np.int64) == np.asarray(y_pred, dtype=np.int64)))


def balanced_accuracy_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    pos_mask = y_true == 1
    neg_mask = y_true == 0
    tpr = float(np.mean(y_pred[pos_mask] == 1)) if pos_mask.any() else 0.5
    tnr = float(np.mean(y_pred[neg_mask] == 0)) if neg_mask.any() else 0.5
    return 0.5 * (tpr + tnr)


def f1_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    return float(2 * precision * recall / (precision + recall + 1e-9))

In [ ]:
def attach_vol_regime(pred_df: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    out = pred_df.copy()
    rolling_vol = frame["rolling_vol"].values.astype(np.float64)
    idx = np.clip(out["anchor"].values.astype(int) - 1, 0, len(rolling_vol) - 1)
    out["rolling_vol"] = rolling_vol[idx]
    q1, q2 = np.nanquantile(out["rolling_vol"], [0.33, 0.67])
    if not np.isfinite(q1) or not np.isfinite(q2) or q1 >= q2:
        out["vol_regime"] = "flat_vol"
    else:
        out["vol_regime"] = pd.cut(
            out["rolling_vol"],
            bins=[-np.inf, q1, q2, np.inf],
            labels=["low_vol", "mid_vol", "high_vol"],
            include_lowest=True,
        ).astype(str)
    return out


def regression_prediction_frame(frame: pd.DataFrame, anchors: np.ndarray, horizon: int, pred_return: np.ndarray) -> pd.DataFrame:
    close = frame["Close"].values.astype(np.float64)
    anchors = np.asarray(anchors, dtype=np.int64)
    last_close = close[anchors - 1]
    true_close = close[anchors + horizon - 1]
    true_return = np.log(np.clip(true_close, 1e-8, None)) - np.log(np.clip(last_close, 1e-8, None))
    pred_close = last_close * np.exp(pred_return)
    out = pd.DataFrame(
        {
            "anchor": anchors,
            "anchor_time": frame.index[anchors],
            "target_time": frame.index[anchors + horizon - 1],
            "last_close": last_close,
            "true_close": true_close,
            "pred_close": pred_close,
            "true_return": true_return,
            "pred_return": pred_return,
        }
    )
    out["abs_price_error"] = np.abs(out["pred_close"] - out["true_close"])
    out["abs_return_error"] = np.abs(out["pred_return"] - out["true_return"])
    out["direction_hit"] = (np.sign(out["pred_return"]) == np.sign(out["true_return"])).astype(np.int8)
    return out


def direction_prediction_frame(frame: pd.DataFrame, anchors: np.ndarray, horizon: int, pred_label: np.ndarray, pred_prob=None) -> pd.DataFrame:
    close = frame["Close"].values.astype(np.float64)
    anchors = np.asarray(anchors, dtype=np.int64)
    last_close = close[anchors - 1]
    true_close = close[anchors + horizon - 1]
    true_return = np.log(np.clip(true_close, 1e-8, None)) - np.log(np.clip(last_close, 1e-8, None))
    true_label = (true_return > 0).astype(np.int8)
    if pred_prob is None:
        pred_prob = pred_label.astype(np.float64)
    out = pd.DataFrame(
        {
            "anchor": anchors,
            "anchor_time": frame.index[anchors],
            "target_time": frame.index[anchors + horizon - 1],
            "last_close": last_close,
            "true_close": true_close,
            "true_return": true_return,
            "true_label": true_label,
            "pred_prob": pred_prob,
            "pred_label": pred_label.astype(np.int8),
        }
    )
    out["correct"] = (out["true_label"] == out["pred_label"]).astype(np.int8)
    return out


def regression_metrics(pred_df: pd.DataFrame):
    return {
        "return_mae": mae(pred_df["pred_return"], pred_df["true_return"]),
        "return_rmse": rmse(pred_df["pred_return"], pred_df["true_return"]),
        "price_mae": mae(pred_df["pred_close"], pred_df["true_close"]),
        "price_rmse": rmse(pred_df["pred_close"], pred_df["true_close"]),
        "direction_acc": float(pred_df["direction_hit"].mean()),
    }


def direction_metrics(pred_df: pd.DataFrame):
    return {
        "accuracy": accuracy_np(pred_df["true_label"], pred_df["pred_label"]),
        "balanced_accuracy": balanced_accuracy_np(pred_df["true_label"], pred_df["pred_label"]),
        "f1": f1_np(pred_df["true_label"], pred_df["pred_label"]),
        "positive_rate_pred": float(np.mean(pred_df["pred_label"])),
        "positive_rate_true": float(np.mean(pred_df["true_label"])),
    }


def regression_baselines(frame: pd.DataFrame, anchors: np.ndarray, horizon: int, seasonal_lag: int):
    close = frame["Close"].values.astype(np.float64)
    anchors = np.asarray(anchors, dtype=np.int64)
    zero_pred = np.zeros(len(anchors), dtype=np.float64)
    last_pred = np.zeros(len(anchors), dtype=np.float64)
    mask_last = anchors >= 2
    one_step = np.log(np.clip(close[anchors[mask_last] - 1], 1e-8, None)) - np.log(np.clip(close[anchors[mask_last] - 2], 1e-8, None))
    last_pred[mask_last] = horizon * one_step

    seasonal_pred = np.zeros(len(anchors), dtype=np.float64)
    valid = (anchors - seasonal_lag - 1 >= 0) & (anchors - seasonal_lag + horizon - 1 < len(close))
    seasonal_anchor = anchors[valid] - seasonal_lag
    seasonal_pred[valid] = (
        np.log(np.clip(close[seasonal_anchor + horizon - 1], 1e-8, None))
        - np.log(np.clip(close[seasonal_anchor - 1], 1e-8, None))
    )

    return {
        "zero_return": regression_prediction_frame(frame, anchors, horizon, zero_pred),
        "last_return": regression_prediction_frame(frame, anchors, horizon, last_pred),
        "seasonal_return": regression_prediction_frame(frame, anchors, horizon, seasonal_pred),
    }


def direction_baselines(frame: pd.DataFrame, train_anchors: np.ndarray, test_anchors: np.ndarray, horizon: int):
    close = frame["Close"].values.astype(np.float64)
    train_last = close[train_anchors - 1]
    train_future = close[train_anchors + horizon - 1]
    train_labels = (np.log(np.clip(train_future, 1e-8, None)) - np.log(np.clip(train_last, 1e-8, None)) > 0).astype(np.int8)
    majority_label = np.int8(1 if train_labels.mean() >= 0.5 else 0)
    majority_pred = np.full(len(test_anchors), majority_label, dtype=np.int8)

    last_sign_pred = np.zeros(len(test_anchors), dtype=np.int8)
    mask_last = test_anchors >= 2
    prev_ret = np.log(np.clip(close[test_anchors[mask_last] - 1], 1e-8, None)) - np.log(np.clip(close[test_anchors[mask_last] - 2], 1e-8, None))
    last_sign_pred[mask_last] = (prev_ret > 0).astype(np.int8)

    return {
        "majority_sign": direction_prediction_frame(frame, test_anchors, horizon, majority_pred),
        "last_sign": direction_prediction_frame(frame, test_anchors, horizon, last_sign_pred),
    }


def predict_regression(model, dataset: Dataset, frame: pd.DataFrame, horizon: int, y_scaler, cfg: CFG):
    loader = make_loader(dataset, shuffle=False, cfg=cfg)
    pred_scaled, anchors = [], []
    model.eval()
    with torch.no_grad():
        for xb, _, anchor, *_ in loader:
            xb = xb.to(device, non_blocking=True)
            with autocast("cuda", enabled=(cfg.amp and device == "cuda")):
                pred = model(xb).detach().cpu().numpy().reshape(-1)
            pred_scaled.append(pred)
            anchors.append(anchor.numpy().reshape(-1))
    pred_return = inverse_scale(np.concatenate(pred_scaled), y_scaler)
    anchors = np.concatenate(anchors)
    return regression_prediction_frame(frame, anchors, horizon, pred_return)


def predict_direction(model, dataset: Dataset, frame: pd.DataFrame, horizon: int, cfg: CFG):
    loader = make_loader(dataset, shuffle=False, cfg=cfg)
    logits, anchors = [], []
    model.eval()
    with torch.no_grad():
        for xb, _, anchor, *_ in loader:
            xb = xb.to(device, non_blocking=True)
            with autocast("cuda", enabled=(cfg.amp and device == "cuda")):
                pred = model(xb).detach().cpu().numpy().reshape(-1)
            logits.append(pred)
            anchors.append(anchor.numpy().reshape(-1))
    logits = np.concatenate(logits)
    prob = 1.0 / (1.0 + np.exp(-logits))
    pred_label = (prob >= 0.5).astype(np.int8)
    anchors = np.concatenate(anchors)
    return direction_prediction_frame(frame, anchors, horizon, pred_label, pred_prob=prob)

## Подготовка данных и фолдов

In [ ]:
raw_1m = load_1m_with_fallback(cfg)
raw_1m["Datetime"] = pd.to_datetime(raw_1m["Datetime"], utc=True)
raw_1m = raw_1m.sort_values("Datetime").reset_index(drop=True)

frames = {}
folds_by_freq = {}

for freq in sorted(set([r["freq"] for r in REGRESSION_RUNS + DIRECTION_RUNS])):
    frame = build_frame(raw_1m, freq)
    frames[freq] = frame
    folds = make_folds(len(frame), freq, cfg)
    folds_by_freq[freq] = folds
    print("\n" + "=" * 100)
    print(f"freq={freq} | shape={frame.shape} | folds={len(folds)}")
    print("date range:", frame.index.min(), "->", frame.index.max())
    display(frame.head())
    display(pd.DataFrame(folds))

In [ ]:
regression_rows = []
regression_predictions = []
direction_rows = []
direction_predictions = []

for run in REGRESSION_RUNS:
    freq = run["freq"]
    horizon = run["horizon"]
    frame = frames[freq]
    folds = folds_by_freq[freq]
    preset = FREQ_PRESETS[freq]
    feature_cols = [c for c in cfg.relative_feature_cols if c in frame.columns]
    close_values = frame["Close"].values.astype(np.float32)

    print("\n" + "#" * 110)
    print(f"REGRESSION | freq={freq} | horizon={horizon}")

    for fold_id, fold in enumerate(folds, start=1):
        train_anchors = make_anchors(0, fold["train_end"], preset["lookback"], horizon)
        val_anchors = make_anchors(fold["train_end"], fold["val_end"], preset["lookback"], horizon)
        test_anchors = make_anchors(fold["val_end"], fold["test_end"], preset["lookback"], horizon)
        if min(len(train_anchors), len(val_anchors), len(test_anchors)) == 0:
            continue

        train_df = frame.iloc[: fold["train_end"]].copy()
        x_scaler = fit_feature_scaler(train_df, feature_cols)
        y_scaler = fit_return_scaler(close_values, train_anchors, horizon)

        for model_name, pred_df in regression_baselines(frame, test_anchors, horizon, preset["seasonal_lag"]).items():
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "regression"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            regression_predictions.append(pred_df)
            row = {"freq": freq, "task": "regression", "horizon": horizon, "fold": fold_id, "model": model_name, "n_samples": len(pred_df)}
            row.update(regression_metrics(pred_df))
            regression_rows.append(row)

        numeric_train = NumericDataset(frame, feature_cols, train_anchors, preset["lookback"], horizon, "regression", x_scaler=x_scaler, y_scaler=y_scaler)
        numeric_val = NumericDataset(frame, feature_cols, val_anchors, preset["lookback"], horizon, "regression", x_scaler=x_scaler, y_scaler=y_scaler)
        numeric_test = NumericDataset(frame, feature_cols, test_anchors, preset["lookback"], horizon, "regression", x_scaler=x_scaler, y_scaler=y_scaler)

        if "numeric_relative" in run["learned_models"]:
            model, hist = fit_model("numeric_relative", "regression", numeric_train, numeric_val, cfg, preset["lookback"], n_features=len(feature_cols))
            pred_df = predict_regression(model, numeric_test, frame, horizon, y_scaler, cfg)
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "regression"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = "numeric_relative"
            regression_predictions.append(pred_df)
            row = {"freq": freq, "task": "regression", "horizon": horizon, "fold": fold_id, "model": "numeric_relative", "n_samples": len(pred_df)}
            row.update(regression_metrics(pred_df))
            regression_rows.append(row)
            if device == "cuda":
                torch.cuda.empty_cache()

        for learned in [m for m in run["learned_models"] if m.startswith("symbolic_")]:
            variant = learned.replace("symbolic_", "")
            token_col = f"token_{variant}"
            token_vocab = fit_token_vocab(train_df[token_col])
            train_ds = TokenDataset(frame, token_col, token_vocab, train_anchors, preset["lookback"], horizon, "regression", y_scaler=y_scaler)
            val_ds = TokenDataset(frame, token_col, token_vocab, val_anchors, preset["lookback"], horizon, "regression", y_scaler=y_scaler)
            test_ds = TokenDataset(frame, token_col, token_vocab, test_anchors, preset["lookback"], horizon, "regression", y_scaler=y_scaler)
            model, hist = fit_model(learned, "regression", train_ds, val_ds, cfg, preset["lookback"], vocab_size=len(token_vocab))
            pred_df = predict_regression(model, test_ds, frame, horizon, y_scaler, cfg)
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "regression"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = learned
            regression_predictions.append(pred_df)
            row = {"freq": freq, "task": "regression", "horizon": horizon, "fold": fold_id, "model": learned, "n_samples": len(pred_df), "vocab_size": len(token_vocab)}
            row.update(regression_metrics(pred_df))
            regression_rows.append(row)
            if device == "cuda":
                torch.cuda.empty_cache()

In [ ]:
for run in DIRECTION_RUNS:
    freq = run["freq"]
    horizon = run["horizon"]
    frame = frames[freq]
    folds = folds_by_freq[freq]
    preset = FREQ_PRESETS[freq]
    feature_cols = [c for c in cfg.relative_feature_cols if c in frame.columns]

    print("\n" + "#" * 110)
    print(f"DIRECTION | freq={freq} | horizon={horizon}")

    for fold_id, fold in enumerate(folds, start=1):
        train_anchors = make_anchors(0, fold["train_end"], preset["lookback"], horizon)
        val_anchors = make_anchors(fold["train_end"], fold["val_end"], preset["lookback"], horizon)
        test_anchors = make_anchors(fold["val_end"], fold["test_end"], preset["lookback"], horizon)
        if min(len(train_anchors), len(val_anchors), len(test_anchors)) == 0:
            continue

        train_df = frame.iloc[: fold["train_end"]].copy()
        x_scaler = fit_feature_scaler(train_df, feature_cols)

        for model_name, pred_df in direction_baselines(frame, train_anchors, test_anchors, horizon).items():
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "direction"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = model_name
            direction_predictions.append(pred_df)
            row = {"freq": freq, "task": "direction", "horizon": horizon, "fold": fold_id, "model": model_name, "n_samples": len(pred_df)}
            row.update(direction_metrics(pred_df))
            direction_rows.append(row)

        numeric_train = NumericDataset(frame, feature_cols, train_anchors, preset["lookback"], horizon, "direction", x_scaler=x_scaler)
        numeric_val = NumericDataset(frame, feature_cols, val_anchors, preset["lookback"], horizon, "direction", x_scaler=x_scaler)
        numeric_test = NumericDataset(frame, feature_cols, test_anchors, preset["lookback"], horizon, "direction", x_scaler=x_scaler)

        if "numeric_relative" in run["learned_models"]:
            model, hist = fit_model("numeric_relative", "direction", numeric_train, numeric_val, cfg, preset["lookback"], n_features=len(feature_cols))
            pred_df = predict_direction(model, numeric_test, frame, horizon, cfg)
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "direction"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = "numeric_relative"
            direction_predictions.append(pred_df)
            row = {"freq": freq, "task": "direction", "horizon": horizon, "fold": fold_id, "model": "numeric_relative", "n_samples": len(pred_df)}
            row.update(direction_metrics(pred_df))
            direction_rows.append(row)
            if device == "cuda":
                torch.cuda.empty_cache()

        for learned in [m for m in run["learned_models"] if m.startswith("symbolic_")]:
            variant = learned.replace("symbolic_", "")
            token_col = f"token_{variant}"
            token_vocab = fit_token_vocab(train_df[token_col])
            train_ds = TokenDataset(frame, token_col, token_vocab, train_anchors, preset["lookback"], horizon, "direction")
            val_ds = TokenDataset(frame, token_col, token_vocab, val_anchors, preset["lookback"], horizon, "direction")
            test_ds = TokenDataset(frame, token_col, token_vocab, test_anchors, preset["lookback"], horizon, "direction")
            model, hist = fit_model(learned, "direction", train_ds, val_ds, cfg, preset["lookback"], vocab_size=len(token_vocab))
            pred_df = predict_direction(model, test_ds, frame, horizon, cfg)
            pred_df = attach_vol_regime(pred_df, frame)
            pred_df["freq"] = freq
            pred_df["task"] = "direction"
            pred_df["horizon"] = horizon
            pred_df["fold"] = fold_id
            pred_df["model"] = learned
            direction_predictions.append(pred_df)
            row = {"freq": freq, "task": "direction", "horizon": horizon, "fold": fold_id, "model": learned, "n_samples": len(pred_df), "vocab_size": len(token_vocab)}
            row.update(direction_metrics(pred_df))
            direction_rows.append(row)
            if device == "cuda":
                torch.cuda.empty_cache()

In [ ]:
regression_df = pd.DataFrame(regression_rows).sort_values(["freq", "horizon", "fold", "model"]).reset_index(drop=True)
direction_df = pd.DataFrame(direction_rows).sort_values(["freq", "horizon", "fold", "model"]).reset_index(drop=True)
regression_pred_df = pd.concat(regression_predictions, ignore_index=True)
direction_pred_df = pd.concat(direction_predictions, ignore_index=True)

reg_baseline = (
    regression_df[regression_df["model"] == "zero_return"][["freq", "horizon", "fold", "price_mae", "price_rmse"]]
    .rename(columns={"price_mae": "baseline_price_mae", "price_rmse": "baseline_price_rmse"})
)
regression_df = regression_df.merge(reg_baseline, on=["freq", "horizon", "fold"], how="left")
regression_df["price_mae_vs_baseline"] = regression_df["price_mae"] / regression_df["baseline_price_mae"]
regression_df["price_rmse_vs_baseline"] = regression_df["price_rmse"] / regression_df["baseline_price_rmse"]

dir_baseline = (
    direction_df[direction_df["model"] == "majority_sign"][["freq", "horizon", "fold", "balanced_accuracy", "accuracy", "f1"]]
    .rename(columns={"balanced_accuracy": "baseline_balanced_accuracy", "accuracy": "baseline_accuracy", "f1": "baseline_f1"})
)
direction_df = direction_df.merge(dir_baseline, on=["freq", "horizon", "fold"], how="left")
direction_df["balanced_accuracy_vs_baseline"] = direction_df["balanced_accuracy"] - direction_df["baseline_balanced_accuracy"]
direction_df["accuracy_vs_baseline"] = direction_df["accuracy"] - direction_df["baseline_accuracy"]
direction_df["f1_vs_baseline"] = direction_df["f1"] - direction_df["baseline_f1"]

regression_summary = (
    regression_df.groupby(["freq", "horizon", "model"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        price_mae_mean=("price_mae", "mean"),
        price_mae_std=("price_mae", "std"),
        price_rmse_mean=("price_rmse", "mean"),
        return_mae_mean=("return_mae", "mean"),
        direction_acc_mean=("direction_acc", "mean"),
        price_mae_vs_baseline_mean=("price_mae_vs_baseline", "mean"),
        price_mae_vs_baseline_std=("price_mae_vs_baseline", "std"),
    )
    .sort_values(["freq", "horizon", "price_mae_mean", "model"])
    .reset_index(drop=True)
)

direction_summary = (
    direction_df.groupby(["freq", "horizon", "model"], as_index=False)
    .agg(
        folds=("fold", "nunique"),
        accuracy_mean=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        balanced_accuracy_vs_baseline_mean=("balanced_accuracy_vs_baseline", "mean"),
        balanced_accuracy_vs_baseline_std=("balanced_accuracy_vs_baseline", "std"),
    )
    .sort_values(["freq", "horizon", "balanced_accuracy_mean", "model"], ascending=[True, True, False, True])
    .reset_index(drop=True)
)

reg_table = regression_summary.assign(
    task="regression",
    primary_metric=regression_summary["price_mae_mean"],
    uncertainty=regression_summary["price_mae_std"],
    delta_vs_baseline=regression_summary["price_mae_vs_baseline_mean"],
)[["task", "freq", "horizon", "model", "folds", "primary_metric", "uncertainty", "delta_vs_baseline"]]

dir_table = direction_summary.assign(
    task="direction",
    primary_metric=direction_summary["balanced_accuracy_mean"],
    uncertainty=direction_summary["balanced_accuracy_std"],
    delta_vs_baseline=direction_summary["balanced_accuracy_vs_baseline_mean"],
)[["task", "freq", "horizon", "model", "folds", "primary_metric", "uncertainty", "delta_vs_baseline"]]

paper_ready_table = pd.concat([reg_table, dir_table], ignore_index=True)

outputs = {
    "regression_fold_metrics.csv": regression_df,
    "direction_fold_metrics.csv": direction_df,
    "regression_summary.csv": regression_summary,
    "direction_summary.csv": direction_summary,
    "paper_ready_table.csv": paper_ready_table,
    "regression_predictions.csv": regression_pred_df,
    "direction_predictions.csv": direction_pred_df,
}

for filename, df in outputs.items():
    path = os.path.join(OUT_DIR, filename)
    df.to_csv(path, index=False)
    print("saved:", path)

display(regression_summary)
display(direction_summary)
display(paper_ready_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

reg_plot = regression_summary.copy()
for label, group in reg_plot.groupby("model"):
    x = [f"{r.freq}\nH={int(r.horizon)}" for r in group.itertuples()]
    axes[0].plot(x, group["price_mae_vs_baseline_mean"], marker="o", label=label)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Regression: MAE vs zero_return")
axes[0].set_ylabel("MAE ratio")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

dir_plot = direction_summary.copy()
for label, group in dir_plot.groupby("model"):
    x = [f"{r.freq}\nH={int(r.horizon)}" for r in group.itertuples()]
    axes[1].plot(x, group["balanced_accuracy_mean"], marker="o", label=label)
axes[1].axhline(0.5, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Direction: balanced accuracy")
axes[1].set_ylabel("Balanced accuracy")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

## Как читать результаты

Для регрессии:
- если `price_mae_vs_baseline_mean < 1.0`, модель лучше `zero_return`
- если стандартное отклонение маленькое, результат более устойчив между фолдами

Для направления:
- если `balanced_accuracy_mean > 0.5`, модель лучше случайного двоичного решения
- если `balanced_accuracy_vs_baseline_mean > 0`, модель лучше `majority_sign`

После запуска этого ноутбука следующий шаг уже не новый эксперимент, а итоговая суммаризация для курсовой:
- краткое описание постановки
- таблица конфигураций
- сравнение baseline и learned models
- выводы по symbolic representation